# Chapter 28: combining factors

Fresh cells winsorize and standardize, neutralize groups, compare linear projection, combine correlated signals, handle missingness and expose full-sample scaling leakage. A known-DGP study then examines sequential processing, composition, tuning and constraints. It performs fresh simulations rather than replaying a table. French factor correlations remain disabled.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). See this guide for run steps, data requirements, and expected outputs.

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch28_multifactor_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

import sys
from pathlib import Path

sys.path.insert(0, str(ROOT / "src"))
print("interpreter:", sys.executable)
print("project root:", ROOT)
import importlib.util


def load_code_module(filename: str):
    """Import a chapter CLI from code/ so the notebook calls exactly the functions the CLI runs."""
    spec = importlib.util.spec_from_file_location(Path(filename).stem, ROOT / "code" / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

## 1. Transformations

For 11 values, 10th/90th percentile clipping changes 1 to 2 and 100 to 10. Compare z-scores and normal scores, retaining their different meanings.

In [ ]:
import numpy as np
import pandas as pd
from quant_course.factor_preprocessing import normal_scores, winsorize_cross_section, zscore_cross_section

x = pd.DataFrame([[1.0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 100]], index=pd.period_range("2026-06", periods=1, freq="M"))
w = winsorize_cross_section(x, lower=0.1, upper=0.9, min_names=10)
table = pd.DataFrame({"raw": x.iloc[0], "winsorized": w.iloc[0], "z raw": zscore_cross_section(x).iloc[0], "z winsorized": zscore_cross_section(w).iloc[0], "normal score": normal_scores(x).iloc[0]})
assert w.iloc[0].tolist() == [2, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10]
table.round(3)

## 2. Neutrality and projection

Two three-name groups have different level shifts. Compare group residuals with projection onto the specified constraints, then raw versus neutral IC. Neutrality concerns signed exposures.

In [ ]:
from quant_course.factor_preprocessing import neutralize
from quant_course.portfolio_construction import project_to_constraints

cols = list("abcdef")
sig = pd.DataFrame([[1.0, -1.0, 0.0, 11.0, 10.0, 9.0]], index=pd.period_range("2026-06", periods=1, freq="M"), columns=cols)
ret = np.array([0.05, 0.03, 0.04, -0.03, -0.04, -0.05])
res = neutralize(sig, groups=pd.Series(list("AAABBB"), index=cols))
A = np.array([[1, 1, 1, 0, 0, 0], [0, 0, 0, 1, 1, 1]], dtype=float)
proj = project_to_constraints(sig.iloc[0].to_numpy(), A)
assert np.allclose(proj, res.iloc[0], atol=1e-12)
print("raw IC", round(np.corrcoef(sig.iloc[0], ret)[0, 1], 4), "| neutral IC", round(np.corrcoef(res.iloc[0], ret)[0, 1], 4))
pd.DataFrame({"residual": res.iloc[0], "projection": proj}).round(4)

## 3. Correlated signals and order

With signal correlation 0.5 and ICs (0.04,0.02), compare equal weights, both orthogonalization orders and the optimal R⁻¹rho direction. The direction still needs the specified normalization.

In [ ]:
from quant_course.factor_preprocessing import composite_ic, optimal_composite_weights, sequential_orthogonal_ic

R = np.array([[1.0, 0.5], [0.5, 1.0]]); rho = [0.04, 0.02]
weights, best = optimal_composite_weights(rho, R)
out = {"only z1": 0.04, "equal weights": composite_ic(rho, R), "z1 then z2": sequential_orthogonal_ic(rho, R, [0, 1]),
       "z2 then z1": sequential_orthogonal_ic(rho, R, [1, 0]), "optimal": best}
assert out["z2 then z1"] > out["equal weights"] > out["z1 then z2"] and abs(best - 0.04) < 1e-12
pd.Series(out).round(4)

## 4. Missingness and scaling leakage

For a name missing factor 1, reweighting factor 2 gives 1 while zero filling gives 0.5. Observation 3 has opposite past-window and full-sample z-score signs because a future value of 20 changes the latter.

In [ ]:
from quant_course.factor_preprocessing import combine_signals, expanding_zscore

idx = pd.period_range("2026-06", periods=1, freq="M")
z = {"f1": pd.DataFrame([[2.0, np.nan]], index=idx, columns=["s1", "s2"]), "f2": pd.DataFrame([[2.0, 1.0]], index=idx, columns=["s1", "s2"])}
rw, _ = combine_signals(z, {"f1": 0.5, "f2": 0.5}, missing="reweight"); zf, _ = combine_signals(z, {"f1": 0.5, "f2": 0.5}, missing="zero")
s = pd.DataFrame({"x": [4.0, 6.0, 8.0, 20.0]}, index=pd.period_range("2026-01", periods=4, freq="M"))
past = expanding_zscore(s, min_periods=3)["x"].iloc[2]; full = ((s["x"] - s["x"].mean()) / s["x"].std(ddof=1)).iloc[2]
assert rw.iloc[0, 1] == 1.0 and zf.iloc[0, 1] == 0.5 and past > 0 > full
print("single-factor name: reweighted", rw.iloc[0, 1], "| zero-filled", zf.iloc[0, 1], "| third obs: expanding z", round(past, 3), "full-sample z", round(full, 3))

## 5. Constructed panel

Inspect processing attenuation, composition order, discovery-selected weights in holdout and constraints. Oracle loadings belong to the known DGP, not observable investment parameters. One path does not validate a tuning procedure.

In [ ]:
ch28 = load_code_module("ch28_multifactor.py")
cons = ch28.constructed_study()
steps = pd.DataFrame({f: {k: v["mean"] for k, v in cons["factor_ic"][f].items() if isinstance(v, dict)} for f in ch28.FACTORS}).T
assert steps.loc["momentum", "raw_pearson"] < steps.loc["momentum", "winsor_pearson"] < steps.loc["momentum", "neutral_winsor_pearson"]
steps[["raw_pearson", "winsor_pearson", "neutral_winsor_pearson", "neutral_winsor_spearman", "true_signal_pearson"]].round(4)

In [ ]:
att = pd.DataFrame(cons["tuning"]["attenuation"]).T[["true", "ratio_raw_to_true", "ratio_winsor_to_true", "ratio_neutral_winsor_to_true"]]
print("predicted raw/true (section 6.2): value %.3f, momentum %.3f, quality %.3f" % tuple(1 / np.sqrt(1 + 0.9 + c * c + 1.28) for c in (-0.5, 0.0, 0.3)))
att.round(3)

In [ ]:
comp = pd.DataFrame({k: {"Spearman": v["spearman"]["mean"], "Pearson": v["pearson"]["mean"]} for k, v in cons["composite_ic"].items()}).T
comp["analytic (Pearson scale)"] = [cons["composite_analytic"]["equal"], cons["composite_analytic"]["oracle_loading_weights"], *cons["composite_analytic"]["sequential"].values()]
assert comp.loc["sequential V>M>Q", "Pearson"] < comp.loc["sequential Q>M>V", "Pearson"]
comp.round(4)

In [ ]:
tune = {k: v for k, v in cons["tuning"].items() if isinstance(v, dict) and "discovery_mean" in v}
print("tuned weights beat equal weights in the hold-out in", round(100 * cons["tuning"]["share_tuned_beats_equal_in_holdout"], 1), "% of", cons["tuning"]["worlds"], "worlds")
assert cons["tuning"]["share_tuned_beats_equal_in_holdout"] < 0.5
pd.DataFrame(tune).T.round(4)

In [ ]:
pd.DataFrame(cons["constraints"]).T.round(4)

## 6. Published factor returns: skipped

Historical correlations, rolling ranges and composite Sharpe are revised aggregate-return descriptions; they neither run here nor test stock PIT signals.

In [ ]:
if RUN_FRENCH:
    real = ch28.real_study()
    print("common months:", real["common_months"], real["first_month"], "to", real["last_month"])
    pd.DataFrame(real["correlation"]).round(2)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
if RUN_FRENCH:
    print("HML-RMW correlation by period:", {k: round(v, 3) for k, v in real["correlation_hml_rmw_by_period"].items()}, "| rolling 60m range:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in real["rolling_60m_hml_rmw"].items()})
    e = real["equal_risk_composite"]
    assert abs(e["composite_sharpe"] - e["formula_sharpe"]) < 0.01 and real["correlation_hml_rmw_by_period"]["1963-07..1999-12"] < 0 < real["correlation_hml_rmw_by_period"]["2000-01..2026-08"]
    pd.Series({"composite Sharpe": e["composite_sharpe"], "formula Sharpe": e["formula_sharpe"], "average component Sharpe": e["average_component_sharpe"], "average pairwise corr": e["average_pairwise_correlation"], **{f"Sharpe {k}": v for k, v in e["component_sharpe"].items()}}).round(3)
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


In [ ]:
print("SKIPPED historical figure gallery; constructed calculations above are freshly computed; French historical images are not reproduced")


## 7. Your turn

1. In a disposable code copy change value/quality correlation to +0.3 and compare orthogonalization order.
2. Set $\rho_2=0.03$ in section 3 and verify the order difference using 28-C01.
3. Try 5th/95th percentile winsorization on constructed inputs; compare momentum Pearson IC and state who should choose the cutoffs, and when.